# 07 — Workflow Config

Define entire extraction workflows in YAML and run them without writing Python.

## The portability problem

In practice, the person who builds an extraction pipeline is rarely the person who runs it in production. A data scientist might prototype in Python, tune the schema and review rules, and validate quality — but the pipeline needs to run inside a Java microservice, a Node.js backend, or a CI/CD job where nobody writes Python.

This creates a handoff problem: the extraction logic lives in Python code, tied to specific imports, class hierarchies, and runtime dependencies. Sharing it means sharing code — which means the receiver needs the same Python version, the same packages, and enough context to understand the pipeline's configuration.

### YAML as the universal interface

DocuFlow solves this with **WorkflowConfig** — a single YAML file that captures the entire extraction workflow:

- **Schema** — field names, types, required/optional, descriptions (the same information as a Pydantic model, but in declarative YAML).
- **Parser and model** — which parser to use, which LLM model string.
- **Extraction settings** — type (text/vision/hybrid), mode (single/multi), temperatures, DPI.
- **Validation rules** — which validators to run, which fields they check.
- **Review rules** — trust-gate thresholds, field presence checks, even LLM reviewers.
- **Privacy settings** — anonymization mode, provider, fail-closed behavior.

A YAML file is human-readable, version-controllable, diffable, and language-agnostic. Anyone can read it, anyone can run it — either through `docuflow run workflow.yaml invoice.pdf` on the command line, or by loading it in any language that can parse YAML and call the DocuFlow API.

### Round-trip serialization

The system works in both directions: you can **build** a pipeline in Python and **export** it to YAML for sharing, or you can **write** a YAML file and **run** it without touching Python. This round-trip guarantee means the YAML is always a faithful representation of the pipeline — not a simplified summary.

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

PDF_PATH = "data/sample_invoice.pdf"

## Define a workflow in Python dict

The simplest approach: pass a plain dict that mirrors the YAML structure.

In [ ]:
from docuflow import load_workflow_config

config_dict = {
    "name": "invoice-extraction",
    "version": "1.0",
    "description": "Extract data from US invoices",
    "schema": {
        "supplier_name": {"type": "str", "required": True, "description": "Supplier company name"},
        "invoice_number": {"type": "str", "required": True, "description": "Invoice reference number"},
        "invoice_date": {"type": "str", "required": True, "description": "Date invoice was issued"},
        "due_date": {"type": "str", "description": "Payment due date"},
        "total": {"type": "float", "required": True, "description": "Grand total including tax"},
        "subtotal": {"type": "float", "description": "Amount before tax"},
        "tax_amount": {"type": "float", "description": "Total tax amount"},
        "payment_terms": {"type": "str", "description": "Payment terms"},
    },
    "parser": "pdfplumber",
    "model": "gemini/gemini-2.5-flash",
    "extraction_mode": "single",
    "validation": [
        {"required_fields": ["supplier_name", "invoice_number", "total"]},
    ],
    "review": [
        {"overall_confidence_below": 0.7},
    ],
}

config = load_workflow_config(config_dict)
print(f"Workflow: {config.name} v{config.version}")
print(f"Schema fields: {list(config.schema_.keys())}")
print(f"Parser: {config.parser}")
print(f"Model: {config.model}")

## Run the workflow

`run_workflow` accepts a dict, a YAML file path, or a `WorkflowConfig` object.

In [ ]:
from docuflow import run_workflow

result = run_workflow(config_dict, PDF_PATH)
print(f"Extracted data:")
for k, v in result.data.items():
    print(f"  {k:<20}: {v}")
print(f"\nTrust-gate rate: {result.confidence:.2f}")
print(f"Needs review: {result.needs_review}")

## YAML file

Write the workflow to a YAML file and load it from disk.

In [ ]:
import yaml, tempfile, os

yaml_content = """
name: invoice-workflow
version: "2.0"
description: Full invoice extraction with validation and review
schema:
  supplier_name: {type: str, required: true, description: "Supplier company"}
  invoice_number: {type: str, required: true, description: "Invoice number"}
  invoice_date: {type: str, required: true, description: "Invoice date"}
  due_date: {type: str, description: "Due date"}
  po_number: {type: str, description: "Purchase order number"}
  total: {type: float, required: true, description: "Total amount"}
  subtotal: {type: float, description: "Subtotal before tax"}
  tax_amount: {type: float, description: "Tax amount"}
  payment_terms: {type: str, description: "Payment terms"}
parser: pdfplumber
model: gemini/gemini-2.5-flash
extraction_mode: single
validation:
  - required_fields: [supplier_name, invoice_number, total]
  - evidence_required: [total]
review:
  - overall_confidence_below: 0.7
  - any_field_confidence_below: 0.5
quality_threshold: 0.7
""".strip()

yaml_path = os.path.join(tempfile.mkdtemp(), "invoice_workflow.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"Saved workflow to: {yaml_path}")
print(f"\nYAML content:")
print(yaml_content)

## Run from YAML file

In [ ]:
result2 = run_workflow(yaml_path, PDF_PATH)
print(f"Results from YAML workflow:")
for k, v in result2.data.items():
    print(f"  {k:<20}: {v}")
print(f"\nTrust-gate rate: {result2.confidence:.2f}")

## WorkflowConfig internals

A `WorkflowConfig` builds Pydantic schemas, validators, and review rules from the YAML definition.

In [ ]:
config2 = load_workflow_config(yaml_path)
schema_class = config2.build_schema()
print(f"Generated schema class: {schema_class.__name__}")
print(f"Fields: {list(schema_class.model_fields.keys())}")

validators = config2.build_validators()
print(f"\nValidators: {[v.__class__.__name__ for v in validators]}")

rules = config2.build_review_rules()
print(f"Review rules: {[r.__class__.__name__ for r in rules]}")

## Export from pipeline

Build a pipeline in Python and export it as portable YAML for sharing or CI/CD.

In [ ]:
from docuflow import DocumentPipeline
from pydantic import BaseModel, Field
from docuflow.validation import RequiredFields
from docuflow.review import OverallConfidenceBelow


class SimpleInvoice(BaseModel):
    supplier: str = Field(description="Supplier name")
    total: float = Field(description="Total amount")
    date: str = Field(description="Invoice date")


pipeline = DocumentPipeline(
    parser="pdfplumber",
    model="gemini/gemini-2.5-flash",
    validators=[RequiredFields(["supplier", "total"])],
    review_rules=[OverallConfidenceBelow(0.7)],
)

exported_yaml = pipeline.export_yaml(
    SimpleInvoice, name="simple-invoice", description="Minimal invoice extraction"
)
print("Exported YAML:")
print(exported_yaml)

## Round-trip test

Take the exported YAML, parse it back, and run it on the same PDF.

In [ ]:
result3 = run_workflow(yaml.safe_load(exported_yaml), PDF_PATH)
print(f"Round-trip result:")
for k, v in result3.data.items():
    print(f"  {k:<20}: {v}")

## Auto mode with vision escalation

For mixed-quality document streams (most files fine, some crumpled scans or handwriting where OCR quietly produces garbage), set `extraction_type: auto`. The pipeline parses with the smart parser, gates on the OCR confidence scores, and re-reads the original file with the vision LLM only when quality is below threshold:

```yaml
extraction_type: auto
escalation:
  min_ocr_score: 0.6
  max_low_confidence_ratio: 0.4
  escalate_to: vision   # or hybrid
```

Escalated results carry `result.escalated` and `result.escalation_reason`, and `result.usage` shows what the (more expensive) vision call actually cost. Escalation is suppressed when a privacy policy is configured, because vision sends raw page images to the LLM, bypassing text anonymization.

## CLI usage

Workflows can also be run from the command line:

```bash
docuflow run workflow.yaml invoice.pdf --output result.json
```

This is useful in CI/CD pipelines where you want to run extraction without writing Python code — just ship the YAML file and invoke the CLI.

---

## From config to microservice

So far we've seen how YAML configs make extraction portable — you can share a file instead of sharing code. But in real-world systems, **document extraction is rarely the entire workflow**. It's one step in a larger pipeline that might be built in Java, Go, TypeScript, or any other language.

Consider a typical document processing workflow:

1. A **Java service** receives uploaded documents and queues them
2. A **Python extraction step** pulls structured data from each document
3. A **Go service** validates the extracted data against business rules
4. A **Node.js API** stores results and notifies downstream consumers

The extraction step needs to be callable from any language, not just Python. The most natural way to achieve this is to wrap the extraction workflow in an **HTTP API** — a language-agnostic interface that any system can call.

### Two deployment modes

DocuFlow provides two ways to serve a workflow config over HTTP:

1. **`docuflow serve`** — Start a local HTTP server directly. Great for development, testing, and single-machine deployments.
2. **`docuflow dockerize`** — Generate a complete Docker deployment directory. The extraction service becomes a self-contained container that can be deployed anywhere Docker runs — Kubernetes, ECS, Cloud Run, or a simple `docker compose up`.

Both modes expose the same three endpoints:
- `GET /health` — readiness check (returns workflow name, version, model, parser)
- `GET /schema` — field definitions (so consumers know what to expect)
- `POST /extract` — upload a file, get back structured data + quality score

### Stateless vs persistent

By default, the containerized service is **stateless** — it processes each request independently with no local storage. This is ideal for horizontal scaling: run N replicas behind a load balancer and every instance is identical.

When you need to persist extraction results, quality logs, or document stores, add `--with-storage` to mount a Docker volume at `/data`. The service writes to this volume, and the data survives container restarts.

## Serve a workflow locally

The `create_app()` function takes a `WorkflowConfig` and returns a FastAPI application. Let's see what the server exposes.

In [ ]:
from docuflow.serve import create_app
from docuflow import load_workflow_config

serve_config = load_workflow_config({
    "name": "invoice-service",
    "version": "1.0",
    "description": "Invoice extraction microservice",
    "schema": {
        "supplier_name": {"type": "str", "required": True, "description": "Supplier company"},
        "invoice_number": {"type": "str", "required": True, "description": "Invoice number"},
        "total": {"type": "float", "required": True, "description": "Total amount"},
    },
    "parser": "pdfplumber",
    "model": "gemini/gemini-2.5-flash",
    "validation": [{"required_fields": ["supplier_name", "total"]}],
    "review": [{"overall_confidence_below": 0.7}],
})

app = create_app(serve_config)
print(f"App title: {app.title}")
print(f"App version: {app.version}")
print(f"\nRoutes:")
for route in app.routes:
    if hasattr(route, "methods"):
        for method in route.methods:
            print(f"  {method:6s} {route.path}")

## Test the endpoints with FastAPI TestClient

We can exercise the health and schema endpoints without starting a real server.

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# Health endpoint — consumers use this for readiness probes
health = client.get("/health").json()
print("GET /health")
for k, v in health.items():
    print(f"  {k}: {v}")

print()

# Schema endpoint — consumers discover what fields to expect
schema = client.get("/schema").json()
print("GET /schema")
print(f"  workflow: {schema['workflow']}")
for name, defn in schema["fields"].items():
    req = " (required)" if defn.get("required") else ""
    print(f"  {name}: {defn['type']}{req} — {defn.get('description', '')}")

## Call the extract endpoint

The `POST /extract` endpoint accepts a file upload and returns structured data plus a quality score. This is the endpoint other services in your workflow would call.

In [ ]:
with open(PDF_PATH, "rb") as f:
    resp = client.post("/extract", files={"file": ("invoice.pdf", f, "application/pdf")})

print(f"Status: {resp.status_code}")
data = resp.json()

print(f"\nExtracted data:")
for k, v in data.get("data", {}).items():
    print(f"  {k:<20}: {v}")

print(f"\nQuality score: {data.get('quality_score', 'N/A')}")
print(f"Quality OK: {data.get('quality_ok', 'N/A')}")
print(f"Trust-gate rate: {data.get('confidence', 'N/A')}")
print(f"Needs review: {data.get('needs_review', 'N/A')}")

## Generate a Docker deployment

`generate_deployment()` creates a self-contained directory with everything needed to build and run the extraction service as a Docker container.

In [ ]:
import tempfile
from docuflow.dockerize import generate_deployment

# First, save a workflow config YAML to use as input
workflow_yaml_path = os.path.join(tempfile.mkdtemp(), "invoice_service.yaml")
with open(workflow_yaml_path, "w") as f:
    yaml.dump({
        "name": "invoice-service",
        "version": "1.0",
        "description": "Invoice extraction microservice",
        "schema": {
            "supplier_name": {"type": "str", "required": True, "description": "Supplier company"},
            "invoice_number": {"type": "str", "required": True, "description": "Invoice number"},
            "total": {"type": "float", "required": True, "description": "Total amount"},
        },
        "parser": "pdfplumber",
        "model": "gemini/gemini-2.5-flash",
    }, f)

# Generate the deployment (stateless — no volume)
deploy_dir = os.path.join(tempfile.mkdtemp(), "deploy")
output = generate_deployment(workflow_yaml_path, deploy_dir)

print(f"Deployment generated in: {output}\n")
print("Files created:")
for f in sorted(os.listdir(output)):
    size = os.path.getsize(os.path.join(str(output), f))
    print(f"  {f:<25} ({size} bytes)")

## Inspect the generated files

Let's look at what each generated file contains.

In [ ]:
# The Dockerfile — minimal Python image with uvicorn
print("=== Dockerfile ===")
with open(os.path.join(str(output), "Dockerfile")) as f:
    print(f.read())

# The server.py — just 4 lines that load config and create the app
print("=== server.py ===")
with open(os.path.join(str(output), "server.py")) as f:
    print(f.read())

# Requirements — docuflow + serve dependencies
print("=== requirements.txt ===")
with open(os.path.join(str(output), "requirements.txt")) as f:
    print(f.read())

In [ ]:
# Docker Compose — stateless mode (no volume)
print("=== docker-compose.yml (stateless) ===")
with open(os.path.join(str(output), "docker-compose.yml")) as f:
    print(f.read())

## Persistent storage mode

When you need to keep extraction results, quality logs, or document stores across container restarts, generate with `with_storage=True`. This adds a Docker volume mounted at `/data`.

In [ ]:
# Generate with persistent storage
deploy_persistent = os.path.join(tempfile.mkdtemp(), "deploy_persistent")
output_persistent = generate_deployment(workflow_yaml_path, deploy_persistent, with_storage=True)

print("=== docker-compose.yml (with storage) ===")
with open(os.path.join(str(output_persistent), "docker-compose.yml")) as f:
    print(f.read())

## Calling the service from other languages

Once the container is running, any language can call it. Here are examples showing how the same extraction service integrates into workflows built in different languages:

**curl** (shell scripts, CI/CD):
```bash
curl -X POST http://localhost:8000/extract \
  -F "file=@invoice.pdf" \
  | jq '.data'
```

**JavaScript / TypeScript** (Node.js, Deno):
```javascript
const form = new FormData();
form.append("file", fs.createReadStream("invoice.pdf"));

const resp = await fetch("http://localhost:8000/extract", {
  method: "POST",
  body: form,
});
const { data, quality_score } = await resp.json();
```

**Java** (Spring, enterprise pipelines):
```java
HttpClient client = HttpClient.newHttpClient();
HttpRequest request = HttpRequest.newBuilder()
    .uri(URI.create("http://localhost:8000/extract"))
    .POST(ofFileUpload(Path.of("invoice.pdf"), "file"))
    .build();
Map<String, Object> result = client.send(request, bodyHandler);
```

**Go** (microservice orchestration):
```go
body := &bytes.Buffer{}
writer := multipart.NewWriter(body)
part, _ := writer.CreateFormFile("file", "invoice.pdf")
io.Copy(part, file)
writer.Close()

resp, _ := http.Post("http://localhost:8000/extract", writer.FormDataContentType(), body)
```

The extraction logic stays in Python where it belongs — close to the LLM, the parsers, and the validation rules. Everything else just calls the HTTP API.

## CLI reference

```bash
# Run a workflow directly
docuflow run workflow.yaml invoice.pdf --output result.json

# Start a local HTTP server
docuflow serve workflow.yaml --port 8000

# Generate a Docker deployment (stateless)
docuflow dockerize workflow.yaml --output ./deploy

# Generate with persistent storage volume
docuflow dockerize workflow.yaml --output ./deploy --with-storage

# Build and run the container
cd deploy
docker compose up --build
```